# Phase C: Full-Scale HeteroGNN Model Training

**Hypothesis:** A heterogeneous GNN that combines FinBERT news embeddings with stock correlation
and sector graph structure will outperform text-only baselines for S&P 500 next-day movement prediction.

**Data:** 1.7M news events, 502 stocks, FinBERT 768-dim embeddings + 3-dim sentiment

**Edge Types:**
1. `news → stock` (relates_to) — article mentions stock
2. `stock ↔ stock` (correlated_with) — Pearson |r| > threshold
3. `stock ↔ stock` (same_sector) — GICS sector membership

**Experiments:**

| ID | Model | Description |
|----|-------|-------------|
| B1 | LR + FinBERT embedding | Text-only baseline |
| B2 | LR + sentiment features | Sentiment-only baseline |
| A1 | GNN: news→stock only | Ablation: no stock-stock edges |
| A2 | GNN: + correlated_with | Ablation: add correlation edges |
| A3 | GNN: + same_sector | Ablation: add sector edges |
| **Full** | **GNN: all 3 edge types** | **Full heterogeneous model** |

In [ ]:
# Phase C: Full-Scale HeteroGNN Model Training
# Environment: Google Colab (T4 GPU) or local macOS

import os, sys, time, random, warnings, gc
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
warnings.filterwarnings('ignore', category=FutureWarning)

# --- Reproducibility ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# --- Environment ---
try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_FOLDER = '/content/drive/MyDrive/GNN\u6d4b\u8bd5'
    os.chdir(DRIVE_FOLDER)
    print(f'Colab: working dir = {DRIVE_FOLDER}')
    !pip install --quiet torch_geometric
except ImportError:
    os.chdir('/Users/heruixi/Desktop/GNN-Testing')
    print(f'Local: working dir = {os.getcwd()}')

for d in ['data/fullscale', 'data/reference', 'plots', 'experiments']:
    os.makedirs(d, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ══════════════════════════════════════════════════════════
# PARAMETERS — All tunable values in one place
# ══════════════════════════════════════════════════════════
PARAMS = {
    # Data paths
    'events_file':  'data/fullscale/sp500_news_events.parquet',
    'emb_file':     'data/fullscale/sp500_news_emb_finbert.npy',
    'sent_file':    'data/fullscale/sp500_news_sentiment_finbert.npy',
    'prices_file':  'data/reference/sp500_5y_prices.csv',
    'sectors_file': 'data/reference/sp500_sectors.csv',
    'mcaps_file':   'data/reference/sp500_market_caps.csv',

    # Graph construction
    'corr_threshold': 0.6,           # |Pearson r| for stock-stock edges
    'corr_cutoff': '2024-12-31',     # prices up to here (no future leakage)

    # Time-series split (strict calendar)
    'train_end': '2024-12-31',
    'val_end':   '2025-06-30',
    # test = everything after val_end

    # GNN architecture
    'hidden_channels': 64,
    'dropout': 0.3,

    # Training
    'lr': 1e-3,
    'weight_decay': 1e-4,
    'batch_size': 2048,
    'num_neighbors': [10, 5],        # per-hop sampling
    'epochs': 100,
    'patience': 15,                  # early stopping
}

print('Parameters:')
for k, v in PARAMS.items():
    print(f'  {k}: {v}')

In [ ]:
# === Load All Data ===
t0 = time.time()

# Events metadata
events = pd.read_parquet(PARAMS['events_file'])
events['date'] = pd.to_datetime(events['date'], utc=True)
print(f'Events: {len(events):,} rows, {events["ticker"].nunique()} tickers')

# Prices
prices = pd.read_csv(PARAMS['prices_file'], index_col=0, parse_dates=True)
print(f'Prices: {prices.shape}')

# Sectors
sector_map = pd.read_csv(PARAMS['sectors_file'], index_col=0).squeeze().to_dict()
unique_sectors = sorted(set(sector_map.values()))
print(f'Sectors: {len(unique_sectors)} unique')

# Market caps
mcaps = pd.read_csv(PARAMS['mcaps_file'], index_col=0).squeeze()

# Canonical ticker list (tickers present in BOTH events and prices)
valid_tickers = sorted(set(events['ticker'].unique()) & set(prices.columns))
ticker_to_id = {t: i for i, t in enumerate(valid_tickers)}
num_stocks = len(valid_tickers)
print(f'Valid tickers: {num_stocks}')

# Embeddings (memory-mapped to save RAM)
emb = np.load(PARAMS['emb_file'], mmap_mode='r')
sent = np.load(PARAMS['sent_file'], mmap_mode='r')
print(f'Embeddings: {emb.shape} {emb.dtype}')
print(f'Sentiment:  {sent.shape} {sent.dtype}')

assert len(events) == emb.shape[0] == sent.shape[0], 'Row count mismatch!'
print(f'\nAll data loaded in {time.time()-t0:.1f}s')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PHASE 1a: NEWS DEDUPLICATION
# Same (date, ticker) → one stock-day record
# Embeddings: mean-pooled  |  Sentiment: mean  |  Label: same return
# ══════════════════════════════════════════════════════════════════
from scipy import sparse

t0_phase1 = time.time()
orig_count = len(events)

print('=' * 60)
print('1a: NEWS DEDUPLICATION')
print('=' * 60)

# Normalize dates to day level for grouping
events['_day'] = events['date'].dt.normalize()
gid = events.groupby(['_day', 'ticker']).ngroup().values
n_groups = int(gid.max()) + 1
counts = np.bincount(gid, minlength=n_groups)

# Sparse averaging matrix: W[g, i] = 1/|group_g| if event i ∈ group g
W = sparse.csr_matrix(
    (np.float32(1.0) / counts[gid].astype(np.float32),
     (gid, np.arange(orig_count))),
    shape=(n_groups, orig_count), dtype=np.float32
)

# Average embeddings (loads ~5GB from mmap → ~1.5GB result)
print('  Averaging embeddings across duplicate events...')
emb = W @ np.asarray(emb[:orig_count], dtype=np.float32)
sent = W @ np.asarray(sent[:orig_count], dtype=np.float32)
gc.collect()

# Keep first metadata row per group
events['_gid'] = gid
events_dedup = events.sort_values('_gid').groupby('_gid').first()
events_dedup['n_merged'] = counts
events = events_dedup.reset_index()

print(f'  {orig_count:,} events → {len(events):,} stock-days '
      f'(-{(1 - len(events)/orig_count) * 100:.1f}%)')
print(f'  Median events/stock-day: {int(np.median(counts))}, '
      f'Max: {counts.max()}, '
      f'Singletons: {(counts == 1).sum():,} ({(counts == 1).mean():.0%})')
print(f'  emb: {emb.shape}, sent: {sent.shape}')

del W, gid, counts, events_dedup
gc.collect()

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PHASE 1b: MARKET-ADJUSTED LABELS
# New label: (stock_return_next - market_return_next) > 0
# Market return = equal-weight mean of all S&P 500 constituents
# (SPY not in prices file; equal-weight is cleaner for our universe)
# ══════════════════════════════════════════════════════════════════
print('\n' + '=' * 60)
print('1b: MARKET-ADJUSTED LABELS')
print('=' * 60)

# Daily returns for all stocks
rets_all = prices.pct_change()
market_ret = rets_all.mean(axis=1)  # equal-weight S&P 500 proxy

# Next-day market return: shift(-1) so index=t maps to return on t+1
mkt_next = market_ret.shift(-1)

# Expand to all calendar dates via ffill
# (weekend event → uses Friday's mkt_next, which points to Monday's return ✓)
cal_range = pd.date_range(prices.index.min(), prices.index.max(), freq='D')
mkt_next_full = mkt_next.reindex(cal_range, method='ffill')

# Look up market return for each event
event_day_naive = events['_day'].dt.tz_localize(None)
events['market_return_next'] = event_day_naive.map(mkt_next_full).values
events['excess_return_next'] = events['return_next'] - events['market_return_next']

# Save old label, create market-adjusted label
events['label_raw'] = events['label']
events['label'] = (events['excess_return_next'] > 0).astype(int)

# Report
valid_mkt = events['market_return_next'].notna()
print(f'  Market data coverage: {valid_mkt.sum():,}/{len(events):,} ({valid_mkt.mean():.1%})')
print(f'  Raw label pos rate:     {events["label_raw"].mean():.4f}')
print(f'  Mkt-adj label pos rate: {events.loc[valid_mkt, "label"].mean():.4f}')

# Noise zone comparison
raw_noise = (events['return_next'].abs() < 0.005).mean()
adj_noise = (events.loc[valid_mkt, 'excess_return_next'].abs() < 0.005).mean()
print(f'  Noise zone (|ret|<0.5%): raw={raw_noise:.1%} → mkt-adj={adj_noise:.1%}')

del rets_all, market_ret, mkt_next, mkt_next_full

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PHASE 1c: MOMENTUM & VOLATILITY FEATURES
# 9 per-event features: 3 windows (5/10/21d) × 3 stats (mean/std/momentum)
# All use T-1 close (shift(1)) — strictly no look-ahead
# ══════════════════════════════════════════════════════════════════
print('\n' + '=' * 60)
print('1c: MOMENTUM & VOLATILITY FEATURES')
print('=' * 60)

rets_stocks = prices[valid_tickers].pct_change()

# Compute rolling features for all (trading_day, ticker) pairs
feat_names = []
feat_dfs = []
for w in [5, 10, 21]:
    # Rolling mean return
    feat_names.append(f'ret_mean_{w}d')
    feat_dfs.append(rets_stocks.rolling(w, min_periods=max(1, w // 2)).mean().shift(1))
    # Rolling std (volatility)
    feat_names.append(f'ret_std_{w}d')
    feat_dfs.append(rets_stocks.rolling(w, min_periods=max(1, w // 2)).std().shift(1))
    # Momentum (cumulative return ≈ sum of log returns)
    feat_names.append(f'momentum_{w}d')
    feat_dfs.append(rets_stocks.rolling(w, min_periods=max(1, w // 2)).sum().shift(1))

print(f'  Features: {feat_names}')

# Stack all features into long-format table: (trading_day, ticker) → 9 features
print('  Building feature lookup table...')
feat_combined = {}
for name, fdf in zip(feat_names, feat_dfs):
    stacked = fdf.stack()
    stacked.index.names = ['_td', 'ticker']
    feat_combined[name] = stacked

feat_long = pd.DataFrame(feat_combined).reset_index()
feat_long['_td'] = pd.to_datetime(feat_long['_td'])
print(f'  Lookup table: {len(feat_long):,} rows')

# Map event dates to nearest prior trading day (handles weekends/holidays)
td_sorted = prices.index.sort_values()
event_day_naive = events['_day'].dt.tz_localize(None)
event_day_np = event_day_naive.values.astype('datetime64[ns]')
idx = td_sorted.searchsorted(event_day_np, side='right') - 1
idx = np.clip(idx, 0, len(td_sorted) - 1)
events['_td'] = td_sorted[idx].values

# Merge features into events
events = events.merge(feat_long, on=['_td', 'ticker'], how='left')

# Extract as numpy array aligned with events
momentum_features = events[feat_names].values.astype(np.float32)
n_complete = (~np.isnan(momentum_features)).all(axis=1).sum()
print(f'  Complete rows: {n_complete:,}/{len(events):,} ({n_complete / len(events):.1%})')

# Fill NaN with 0 (events before enough price history)
momentum_features = np.nan_to_num(momentum_features, nan=0.0)

# Clean up temp columns
events = events.drop(columns=feat_names + ['_td', '_day', '_gid'], errors='ignore')
del feat_dfs, feat_combined, feat_long, rets_stocks
gc.collect()

# ── PHASE 1 SUMMARY ──────────────────────────────────────────
print(f'\n{"=" * 60}')
print(f'PHASE 1 COMPLETE ({time.time() - t0_phase1:.1f}s)')
print(f'{"=" * 60}')
print(f'  Events: {orig_count:,} → {len(events):,} stock-days')
print(f'  Embeddings: {emb.shape}')
print(f'  Sentiment:  {sent.shape}')
print(f'  Momentum:   {momentum_features.shape}')
print(f'  Label: market-adjusted (label_raw available for ablation)')

In [ ]:
# === Build Heterogeneous Graph ===
from torch_geometric.data import HeteroData
import torch_geometric.transforms as T

t0 = time.time()
data = HeteroData()

# --- Filter events to valid tickers ---
valid_mask = events['ticker'].isin(ticker_to_id).values
if valid_mask.sum() < len(events):
    print(f'Filtering: {len(events):,} -> {valid_mask.sum():,} events (removed {(~valid_mask).sum():,} with unknown tickers)')
    events = events[valid_mask].reset_index(drop=True)
    # Keep corresponding embedding/feature rows
    valid_idx = np.where(valid_mask)[0]
else:
    valid_idx = None  # use all rows
    print(f'All {len(events):,} events have valid tickers')

num_news = len(events)

# --- 1. News node features: FinBERT (768) + sentiment (3) + momentum (9) = 780-dim ---
print('Building news features...')
if valid_idx is not None:
    news_x = np.concatenate([emb[valid_idx], sent[valid_idx], momentum_features[valid_idx]], axis=1)
else:
    news_x = np.concatenate([emb[:], sent[:], momentum_features[:]], axis=1)
data['news'].x = torch.from_numpy(news_x.astype(np.float32))
del news_x; gc.collect()
print(f'  news.x: {data["news"].x.shape} {data["news"].x.dtype}')

# --- 2. Stock node features: GICS one-hot (11) + log market cap (1) = 12-dim ---
sector_to_idx = {s: i for i, s in enumerate(unique_sectors)}
stock_feat = []
for tk in valid_tickers:
    oh = [0.0] * len(unique_sectors)
    s = sector_map.get(tk, 'Unknown')
    if s in sector_to_idx:
        oh[sector_to_idx[s]] = 1.0
    cap = mcaps.get(tk, 0) if hasattr(mcaps, 'get') else mcaps.loc[tk] if tk in mcaps.index else 0
    log_cap = np.log1p(max(float(cap), 0)) / 30.0
    stock_feat.append(oh + [log_cap])
data['stock'].x = torch.tensor(stock_feat, dtype=torch.float32)
print(f'  stock.x: {data["stock"].x.shape}')

# --- 3. Edge type 1: news -> stock (relates_to) ---
news_src = torch.arange(num_news, dtype=torch.long)
stock_dst = torch.tensor(events['ticker'].map(ticker_to_id).values, dtype=torch.long)
data['news', 'relates_to', 'stock'].edge_index = torch.stack([news_src, stock_dst])
print(f'  news->stock edges: {num_news:,}')

# --- 4. Edge type 2: stock <-> stock (correlated_with) ---
# Use prices up to corr_cutoff to avoid future leakage
cutoff = pd.Timestamp(PARAMS['corr_cutoff'])
prices_train = prices.loc[prices.index <= cutoff, valid_tickers]
rets = prices_train.pct_change(fill_method=None).iloc[1:]
corr_mat = rets.corr().values
thr = PARAMS['corr_threshold']
mask_corr = np.abs(corr_mat) > thr
np.fill_diagonal(mask_corr, False)
c_src, c_dst = np.where(mask_corr)
data['stock', 'correlated_with', 'stock'].edge_index = torch.tensor(
    np.stack([c_src, c_dst]), dtype=torch.long
)
print(f'  stock<->stock (corr |r|>{thr}) edges: {len(c_src):,}')

# --- 5. Edge type 3: stock <-> stock (same_sector) ---
from collections import defaultdict
sec_groups = defaultdict(list)
for tk in valid_tickers:
    sec_groups[sector_map.get(tk, 'Unknown')].append(ticker_to_id[tk])
sec_edges = []
for members in sec_groups.values():
    for i in range(len(members)):
        for j in range(i + 1, len(members)):
            sec_edges.append([members[i], members[j]])
            sec_edges.append([members[j], members[i]])
data['stock', 'same_sector', 'stock'].edge_index = torch.tensor(
    sec_edges, dtype=torch.long
).t()
print(f'  stock<->stock (sector) edges: {len(sec_edges):,}')

# --- 6. Labels & time-series split ---
data['news'].y = torch.tensor(events['label'].values, dtype=torch.long)

dates = events['date']
train_end = pd.Timestamp(PARAMS['train_end'], tz='UTC')
val_end = pd.Timestamp(PARAMS['val_end'], tz='UTC')
data['news'].train_mask = torch.tensor((dates <= train_end).values)
data['news'].val_mask   = torch.tensor(((dates > train_end) & (dates <= val_end)).values)
data['news'].test_mask  = torch.tensor((dates > val_end).values)

n_tr = data['news'].train_mask.sum().item()
n_va = data['news'].val_mask.sum().item()
n_te = data['news'].test_mask.sum().item()
print(f'  Split: train={n_tr:,} | val={n_va:,} | test={n_te:,}')

# --- 7. Add reverse edges (stock -> news) for bidirectional message passing ---
data = T.ToUndirected()(data)

print(f'\nGraph built in {time.time()-t0:.1f}s')
print(data)

In [ ]:
# === Graph Validation ===
print('=== Node Summary ===')
for nt in data.node_types:
    print(f'  {nt}: {data[nt].num_nodes:,} nodes, features={data[nt].x.shape[1]}')

print('\n=== Edge Summary ===')
for et in data.edge_types:
    print(f'  {et}: {data[et].num_edges:,} edges')

print('\n=== Label Distribution ===')
for name, mask in [('train', data['news'].train_mask),
                    ('val',   data['news'].val_mask),
                    ('test',  data['news'].test_mask)]:
    y = data['news'].y[mask]
    pos = y.sum().item()
    tot = len(y)
    print(f'  {name}: {tot:,} samples, {pos/tot:.1%} positive')

# Sanity checks
assert data['news'].x.shape[0] == data['news'].y.shape[0], 'news features vs labels mismatch'
assert data['news'].train_mask.sum() + data['news'].val_mask.sum() + data['news'].test_mask.sum() == data['news'].num_nodes, 'split masks do not cover all nodes'
# edge_index row 0 = source (news), row 1 = destination (stock)
assert data['news', 'relates_to', 'stock'].edge_index[0].max() < data['news'].num_nodes, 'news src index OOB'
assert data['news', 'relates_to', 'stock'].edge_index[1].max() < data['stock'].num_nodes, 'stock dst index OOB'
print('\nAll sanity checks passed.')

In [ ]:
# === Baselines: LR + XGBoost with new features ===
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import roc_auc_score, accuracy_score

results = []  # collect all experiment results

train_mask_np = data['news'].train_mask.numpy()
val_mask_np   = data['news'].val_mask.numpy()
test_mask_np  = data['news'].test_mask.numpy()
labels = data['news'].y.numpy()

train_idx = np.where(train_mask_np)[0]
val_idx   = np.where(val_mask_np)[0]
test_idx  = np.where(test_mask_np)[0]

# Prepare feature arrays (filtered to valid tickers, aligned with graph)
# emb/sent/momentum_features are pre-dedup arrays; after build-graph filtering,
# we need to use the same valid_idx if it was applied
if valid_idx is not None:
    _emb = emb[valid_idx].astype(np.float32)
    _sent = sent[valid_idx].astype(np.float32)
    _mom = momentum_features[valid_idx]
else:
    _emb = emb[:len(events)].astype(np.float32) if not isinstance(emb, np.ndarray) else emb.astype(np.float32)
    _sent = sent[:len(events)].astype(np.float32) if not isinstance(sent, np.ndarray) else sent.astype(np.float32)
    _mom = momentum_features


def run_lr_baseline(name, get_features_fn):
    """Train SGD logistic regression in mini-batches, evaluate on val/test."""
    print(f'\n--- {name} ---')
    t0 = time.time()

    clf = SGDClassifier(
        loss='log_loss', random_state=SEED,
        learning_rate='optimal',
        alpha=1e-4, max_iter=1,
    )

    CHUNK = 50000
    N_EPOCHS = 5

    for epoch in range(N_EPOCHS):
        shuffled = train_idx.copy()
        np.random.shuffle(shuffled)
        for i in range(0, len(shuffled), CHUNK):
            idx = shuffled[i:i+CHUNK]
            X = get_features_fn(idx)
            y = labels[idx]
            clf.partial_fit(X, y, classes=[0, 1])
        if (epoch + 1) % 2 == 0:
            X_val = get_features_fn(val_idx)
            val_score = clf.decision_function(X_val)
            val_auc = roc_auc_score(labels[val_idx], val_score)
            print(f'  Epoch {epoch+1}/{N_EPOCHS}: val AUC = {val_auc:.4f}')

    # Final evaluation
    for split_name, idx in [('Val', val_idx), ('Test', test_idx)]:
        X = get_features_fn(idx)
        score = clf.decision_function(X)
        pred = (score >= 0).astype(int)
        auc = roc_auc_score(labels[idx], score)
        acc = accuracy_score(labels[idx], pred)
        print(f'  {split_name}: AUC={auc:.4f}, Acc={acc:.4f}')
        if split_name == 'Val':
            val_auc, val_acc = auc, acc
        else:
            test_auc, test_acc = auc, acc

    elapsed = time.time() - t0
    results.append({
        'experiment': name, 'val_auc': val_auc, 'val_acc': val_acc,
        'test_auc': test_auc, 'test_acc': test_acc, 'time_s': elapsed,
    })
    print(f'  Time: {elapsed:.1f}s')


# B1: LR + FinBERT embeddings (768-dim)
run_lr_baseline('B1: LR + FinBERT', lambda idx: _emb[idx])

# B2: LR + sentiment only (3-dim)
run_lr_baseline('B2: LR + Sentiment', lambda idx: _sent[idx])

# B3: LR + sentiment + momentum (3 + 9 = 12-dim) — NEW
run_lr_baseline('B3: LR + Sent+Momentum',
                lambda idx: np.concatenate([_sent[idx], _mom[idx]], axis=1))

# B4: LR + momentum only (9-dim) — NEW (test: is signal in price, not NLP?)
run_lr_baseline('B4: LR + Momentum only', lambda idx: _mom[idx])

# B5: XGBoost (sentiment + momentum) — NEW
print('\n--- B5: XGBoost (Sent+Momentum) ---')
t0 = time.time()
try:
    from xgboost import XGBClassifier
    X_train_xgb = np.concatenate([_sent[train_idx], _mom[train_idx]], axis=1)
    X_val_xgb = np.concatenate([_sent[val_idx], _mom[val_idx]], axis=1)
    X_test_xgb = np.concatenate([_sent[test_idx], _mom[test_idx]], axis=1)

    xgb = XGBClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        subsample=0.8, random_state=SEED, eval_metric='auc',
        tree_method='hist',  # fast on large datasets
    )
    xgb.fit(X_train_xgb, labels[train_idx],
            eval_set=[(X_val_xgb, labels[val_idx])], verbose=False)
    val_pred_xgb = xgb.predict_proba(X_val_xgb)[:, 1]
    test_pred_xgb = xgb.predict_proba(X_test_xgb)[:, 1]
    val_auc = roc_auc_score(labels[val_idx], val_pred_xgb)
    test_auc = roc_auc_score(labels[test_idx], test_pred_xgb)
    val_acc = accuracy_score(labels[val_idx], (val_pred_xgb >= 0.5).astype(int))
    test_acc = accuracy_score(labels[test_idx], (test_pred_xgb >= 0.5).astype(int))
    elapsed = time.time() - t0
    print(f'  Val: AUC={val_auc:.4f}, Acc={val_acc:.4f}')
    print(f'  Test: AUC={test_auc:.4f}, Acc={test_acc:.4f}')
    print(f'  Time: {elapsed:.1f}s')
    results.append({
        'experiment': 'B5: XGBoost (Sent+Mom)', 'val_auc': val_auc, 'val_acc': val_acc,
        'test_auc': test_auc, 'test_acc': test_acc, 'time_s': elapsed,
    })
except ImportError:
    print('  XGBoost not available, skipping. Install with: pip install xgboost')

# Print summary
print('\n' + '='*70)
print('BASELINE RESULTS (market-adjusted labels, deduped data)')
print('='*70)
for r in results:
    print(f"{r['experiment']:30s} | Val AUC: {r['val_auc']:.4f} | Test AUC: {r['test_auc']:.4f}")

In [ ]:
# === GNN Model Definition + Full-Batch Training Utilities ===
# Full-batch: load entire graph to GPU (feasible on A100 80GB)
# Advantage: no sampling noise, sees all edges every epoch, no torch-sparse needed

from torch_geometric.nn import SAGEConv, to_hetero


class GNN(torch.nn.Module):
    """2-layer GraphSAGE. Converted to heterogeneous via to_hetero()."""
    def __init__(self, hidden_channels, dropout=0.3):
        super().__init__()
        self.conv1 = SAGEConv((-1, -1), hidden_channels)
        self.conv2 = SAGEConv((-1, -1), hidden_channels)
        self.lin = torch.nn.Linear(hidden_channels, 1)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index).relu()
        return self.lin(x)


def create_ablation_graph(full_data, edge_types_to_keep):
    """Create a HeteroData copy with only the specified forward edge types."""
    from torch_geometric.data import HeteroData
    import torch_geometric.transforms as T

    ab = HeteroData()
    ab['news'].x = full_data['news'].x
    ab['stock'].x = full_data['stock'].x
    ab['news'].y = full_data['news'].y
    ab['news'].train_mask = full_data['news'].train_mask
    ab['news'].val_mask = full_data['news'].val_mask
    ab['news'].test_mask = full_data['news'].test_mask

    for et in edge_types_to_keep:
        ab[et].edge_index = full_data[et].edge_index

    ab = T.ToUndirected()(ab)
    return ab


def run_gnn_experiment(name, graph_data, params):
    """Full-batch GNN experiment: entire graph on GPU, no sampling."""
    print(f'\n{"="*60}')
    print(f'Experiment: {name}')
    print(f'Edge types: {graph_data.edge_types}')
    print(f'{"="*60}')
    t0 = time.time()

    # Reset seeds
    torch.manual_seed(SEED)
    np.random.seed(SEED)

    # Create model for this graph's metadata
    base_model = GNN(params['hidden_channels'], params['dropout'])
    model = to_hetero(base_model, graph_data.metadata(), aggr='mean').to(device)

    # Move entire graph to GPU (float16 news -> float32 on GPU)
    gd = graph_data.clone()
    gd['news'].x = gd['news'].x.float()  # float16 -> float32
    gd = gd.to(device)

    mem_gb = torch.cuda.memory_allocated() / 1e9
    print(f'  GPU memory after loading graph: {mem_gb:.1f} GB')

    optimizer = torch.optim.Adam(model.parameters(), lr=params['lr'],
                                 weight_decay=params['weight_decay'])
    criterion = torch.nn.BCEWithLogitsLoss()

    train_mask = gd['news'].train_mask
    val_mask = gd['news'].val_mask
    test_mask = gd['news'].test_mask

    # Training with early stopping
    best_val_auc = 0
    best_state = None
    stale = 0

    for epoch in range(1, params['epochs'] + 1):
        # --- Train ---
        model.train()
        optimizer.zero_grad()
        out = model(gd.x_dict, gd.edge_index_dict)
        pred = out['news'][train_mask].squeeze(-1)
        target = gd['news'].y[train_mask].float()
        loss = criterion(pred, target)
        loss.backward()
        optimizer.step()

        # --- Eval ---
        model.eval()
        with torch.no_grad():
            out = model(gd.x_dict, gd.edge_index_dict)

            val_pred = out['news'][val_mask].squeeze(-1).sigmoid().cpu().numpy()
            val_target = gd['news'].y[val_mask].cpu().numpy()
            val_auc = roc_auc_score(val_target, val_pred)
            val_acc = accuracy_score(val_target, (val_pred >= 0.5).astype(int))

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            stale = 0
        else:
            stale += 1

        if epoch % 5 == 0 or stale == 0:
            print(f'  Epoch {epoch:3d} | Loss {loss.item():.4f} | Val AUC {val_auc:.4f} | Acc {val_acc:.4f}'
                  f'{" *" if stale == 0 else ""}')

        if stale >= params['patience']:
            print(f'  Early stop at epoch {epoch}')
            break

    # --- Test with best model ---
    if best_state:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    model.eval()
    with torch.no_grad():
        out = model(gd.x_dict, gd.edge_index_dict)
        test_pred = out['news'][test_mask].squeeze(-1).sigmoid().cpu().numpy()
        test_target = gd['news'].y[test_mask].cpu().numpy()
        test_auc = roc_auc_score(test_target, test_pred)
        test_acc = accuracy_score(test_target, (test_pred >= 0.5).astype(int))

    elapsed = time.time() - t0
    print(f'\n  Best Val AUC: {best_val_auc:.4f} | Test AUC: {test_auc:.4f} | Test Acc: {test_acc:.4f}')
    print(f'  Time: {elapsed:.1f}s')

    results.append({
        'experiment': name, 'val_auc': best_val_auc, 'val_acc': val_acc,
        'test_auc': test_auc, 'test_acc': test_acc, 'time_s': elapsed,
    })

    # Save checkpoint
    ckpt_path = f'experiments/{name.replace(" ", "_").replace(":", "")}.pt'
    torch.save(best_state, ckpt_path)
    print(f'  Checkpoint: {ckpt_path}')

    # Free GPU memory for next experiment
    del gd, out
    torch.cuda.empty_cache()

    return model


print('GNN model and utilities defined (full-batch mode).')

In [ ]:
# === Run All GNN Experiments ===

# Define ablation configurations (forward edge types only; reverse added by ToUndirected)
ABLATIONS = {
    'A1: news->stock only': [
        ('news', 'relates_to', 'stock'),
    ],
    'A2: + correlation': [
        ('news', 'relates_to', 'stock'),
        ('stock', 'correlated_with', 'stock'),
    ],
    'A3: + sector': [
        ('news', 'relates_to', 'stock'),
        ('stock', 'same_sector', 'stock'),
    ],
    'Full: all edges': [
        ('news', 'relates_to', 'stock'),
        ('stock', 'correlated_with', 'stock'),
        ('stock', 'same_sector', 'stock'),
    ],
}

for name, edge_types in ABLATIONS.items():
    ab_data = create_ablation_graph(data, edge_types)
    run_gnn_experiment(name, ab_data, PARAMS)
    gc.collect()

In [ ]:
# === Selective AUC Analysis ===
# Test: even if full AUC ≈ 0.50, is there signal in the tails?
# For each method, take the top-K% most confident predictions and compute AUC.
# Go criterion: Full AUC > 0.52 OR Top-10% AUC > 0.54

from sklearn.linear_model import SGDClassifier
from sklearn.metrics import roc_auc_score
from scipy.special import expit

print('='*70)
print('SELECTIVE AUC ANALYSIS')
print('='*70)


def selective_auc(y_true, y_score, coverages=[0.05, 0.10, 0.20, 0.50]):
    """Compute AUC on highest-confidence subsets."""
    confidence = np.abs(y_score - 0.5)
    sorted_idx = np.argsort(-confidence)
    out = {}
    for cov in coverages:
        k = max(int(len(y_true) * cov), 100)
        sel = sorted_idx[:k]
        if len(np.unique(y_true[sel])) < 2:
            out[f'AUC@{int(cov*100)}%'] = float('nan')
        else:
            out[f'AUC@{int(cov*100)}%'] = roc_auc_score(y_true[sel], y_score[sel])
    return out


sel_results = []
test_labels = labels[test_idx]

# --- Re-train baselines, save test probabilities ---
baseline_configs = [
    ('B1: LR+FinBERT', lambda idx: _emb[idx]),
    ('B2: LR+Sent',    lambda idx: _sent[idx]),
    ('B3: LR+Sent+Mom', lambda idx: np.concatenate([_sent[idx], _mom[idx]], axis=1)),
    ('B4: LR+Mom',     lambda idx: _mom[idx]),
]

for name, feat_fn in baseline_configs:
    np.random.seed(SEED)
    clf = SGDClassifier(loss='log_loss', random_state=SEED,
                        learning_rate='optimal', alpha=1e-4, max_iter=1)
    CHUNK = 50000
    for epoch in range(5):
        shuffled = train_idx.copy()
        np.random.shuffle(shuffled)
        for i in range(0, len(shuffled), CHUNK):
            idx = shuffled[i:i+CHUNK]
            clf.partial_fit(feat_fn(idx), labels[idx], classes=[0, 1])
    test_prob = expit(clf.decision_function(feat_fn(test_idx)))
    full_auc = roc_auc_score(test_labels, test_prob)
    sel = selective_auc(test_labels, test_prob)
    row = {'method': name, 'Full AUC': full_auc}
    row.update(sel)
    sel_results.append(row)

# B5: XGBoost
try:
    from xgboost import XGBClassifier
    np.random.seed(SEED)
    xgb_m = XGBClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        subsample=0.8, random_state=SEED, eval_metric='auc', tree_method='hist')
    X_tr = np.concatenate([_sent[train_idx], _mom[train_idx]], axis=1)
    X_te = np.concatenate([_sent[test_idx], _mom[test_idx]], axis=1)
    xgb_m.fit(X_tr, labels[train_idx], verbose=False)
    test_prob = xgb_m.predict_proba(X_te)[:, 1]
    full_auc = roc_auc_score(test_labels, test_prob)
    sel = selective_auc(test_labels, test_prob)
    row = {'method': 'B5: XGB', 'Full AUC': full_auc}
    row.update(sel)
    sel_results.append(row)
except ImportError:
    print('  XGBoost not available')

# --- GNN selective AUC (from saved checkpoints) ---
gnn_names = {
    'A1: news->stock only': 'A1_news-stock_only',
    'Full: all edges': 'Full_all_edges',
}
for display_name, ckpt_name in gnn_names.items():
    ckpt_path = f'experiments/{ckpt_name}.pt'
    try:
        if not os.path.exists(ckpt_path):
            continue
        state = torch.load(ckpt_path, map_location='cpu', weights_only=True)

        # Determine which graph to use
        if 'Full' in display_name:
            gnn_data = data
        else:
            edge_types = [('news', 'relates_to', 'stock')]
            gnn_data = create_ablation_graph(data, edge_types)

        base_m = GNN(PARAMS['hidden_channels'], PARAMS['dropout'])
        het_m = to_hetero(base_m, gnn_data.metadata(), aggr='mean').to(device)
        het_m.load_state_dict({k: v.to(device) for k, v in state.items()})
        het_m.eval()

        gd = gnn_data.clone()
        gd['news'].x = gd['news'].x.float()
        gd = gd.to(device)
        with torch.no_grad():
            out_g = het_m(gd.x_dict, gd.edge_index_dict)
            gnn_prob = out_g['news'][test_mask_np].squeeze(-1).sigmoid().cpu().numpy()

        full_auc = roc_auc_score(test_labels, gnn_prob)
        sel = selective_auc(test_labels, gnn_prob)
        row = {'method': f'GNN {display_name.split(":")[0]}', 'Full AUC': full_auc}
        row.update(sel)
        sel_results.append(row)
        print(f'  Loaded GNN checkpoint: {ckpt_path}')

        del gd, out_g
        torch.cuda.empty_cache()
    except Exception as e:
        print(f'  GNN {display_name} failed: {e}')

# --- Summary Table ---
print(f'\n{"Method":<20} {"Full":>8} {"@50%":>8} {"@20%":>8} {"@10%":>8} {"@5%":>8}')
print('-'*62)
for r in sel_results:
    print(f"{r['method']:<20} {r['Full AUC']:>8.4f} "
          f"{r.get('AUC@50%', float('nan')):>8.4f} "
          f"{r.get('AUC@20%', float('nan')):>8.4f} "
          f"{r.get('AUC@10%', float('nan')):>8.4f} "
          f"{r.get('AUC@5%', float('nan')):>8.4f}")

# --- Go/Stop Assessment ---
print(f'\n{"="*62}')
print('GO/STOP ASSESSMENT (with selective AUC)')
print(f'{"="*62}')
max_full = max(r['Full AUC'] for r in sel_results)
max_10 = max(r.get('AUC@10%', 0) for r in sel_results if not np.isnan(r.get('AUC@10%', 0)))
max_5  = max(r.get('AUC@5%', 0) for r in sel_results if not np.isnan(r.get('AUC@5%', 0)))
print(f'  Max Full AUC:  {max_full:.4f}  (Go threshold: > 0.52)')
print(f'  Max AUC@10%:   {max_10:.4f}  (Go threshold: > 0.54)')
print(f'  Max AUC@5%:    {max_5:.4f}')
go = max_full > 0.52 or max_10 > 0.54
print(f'\n  VERDICT: {"GO — proceed to Phase 2+3" if go else "STOP — signal not found"}')


In [ ]:
# ══════════════════════════════════════════════════════════
# SETUP: OpenRouter API Key (run before Phase 2a)
# ══════════════════════════════════════════════════════════
# Replace 'sk-or-...' with your actual OpenRouter API key.
# To switch to gpt-4o: change model in Cell 14 to "openai/gpt-4o" (~$7.50)
# Current: gpt-4o-mini (~$0.45)

import os
os.environ['OPENAI_API_KEY'] = 'sk-or-...'  # ← 填入您的 OpenRouter key

In [ ]:
# ══════════════════════════════════════════════════════════
# PHASE 2a: GPT-4o-mini Structured Output on Dev-Holdout
# ══════════════════════════════════════════════════════════
# Purpose: Test if LLM impact prediction provides signal beyond FinBERT sentiment.
# FinBERT sentiment (pos/neg/neu) ≈ random for return prediction.
# LLM might add value via: impact_level, reasoning_type — things FinBERT can't do.
#
# Dev-holdout: 2023-Q4 (within training period, never touches val/test)
# Cost estimate: ~$0.15-0.45 for 7K samples
# ══════════════════════════════════════════════════════════

import json as json_mod

# --- Reload original events (with titles) ---
events_orig = pd.read_parquet(PARAMS['events_file'])
events_orig['date'] = pd.to_datetime(events_orig['date'], utc=True)
print(f'Original events: {len(events_orig):,}')

# Dev-holdout: 2023-Q4
dev_mask = (events_orig['date'] >= '2023-10-01') & (events_orig['date'] < '2024-01-01')
dev_events = events_orig[dev_mask].copy()
print(f'Dev-holdout (2023-Q4): {len(dev_events):,} events, {dev_events["ticker"].nunique()} tickers')

# Sample ~7K events (or 10% if fewer)
SAMPLE_N = min(7000, len(dev_events) // 10)
np.random.seed(SEED)
sample_idx = np.random.choice(len(dev_events), size=SAMPLE_N, replace=False)
dev_sample = dev_events.iloc[sample_idx].reset_index(drop=True)

# Keep original parquet indices for FinBERT feature alignment
dev_sample['_orig_idx'] = dev_events.index[sample_idx]
print(f'Sample: {len(dev_sample):,} events')
print(f'  Date range: {dev_sample["date"].min()} → {dev_sample["date"].max()}')
print(f'  Label pos rate: {dev_sample["label"].mean():.4f}')

# --- GPT-4o-mini Structured Output ---
CACHE_FILE = 'data/fullscale/llm_dev_holdout_results.json'

if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE) as f:
        llm_cache = json_mod.load(f)
    print(f'\nLoaded cached LLM results: {len(llm_cache)} entries')
    if len(llm_cache) >= len(dev_sample):
        llm_results = llm_cache[:len(dev_sample)]
        print('Cache complete — skipping API calls')
    else:
        print(f'Cache partial ({len(llm_cache)}/{len(dev_sample)}) — resuming...')
        llm_results = llm_cache
else:
    llm_results = []
    print('\nNo cache found — starting fresh')

if len(llm_results) < len(dev_sample):
    !pip install --quiet openai
    from openai import OpenAI
    client = OpenAI(base_url="https://openrouter.ai/api/v1")  # OpenRouter

    SYSTEM_PROMPT = (
        "You are a financial analyst. Analyze the news headline's potential impact "
        "on the mentioned stock's price over the next trading day. "
        "Consider: Is this material news (earnings, FDA, merger) or routine coverage? "
        "Respond in the required JSON format."
    )

    SCHEMA = {
        "name": "news_impact",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "impact_level": {"type": "string", "enum": ["high", "medium", "low"]},
                "direction": {"type": "string", "enum": ["positive", "negative", "neutral"]},
                "confidence": {"type": "number"},
                "reasoning_type": {"type": "string", "enum": ["earnings", "macro", "sentiment", "technical", "other"]}
            },
            "required": ["impact_level", "direction", "confidence", "reasoning_type"],
            "additionalProperties": False
        }
    }

    start_from = len(llm_results)
    t0 = time.time()
    errors = 0

    for i in range(start_from, len(dev_sample)):
        row = dev_sample.iloc[i]
        title = str(row.get('title', ''))
        ticker = row['ticker']

        try:
            resp = client.chat.completions.create(
                model="openai/gpt-4o-mini",
                response_format={"type": "json_schema", "json_schema": SCHEMA},
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": f"Stock: {ticker}\nHeadline: {title}"}
                ],
                temperature=0,
                max_tokens=80,
            )
            result = json_mod.loads(resp.choices[0].message.content)
        except Exception as e:
            result = {"impact_level": "low", "direction": "neutral",
                      "confidence": 0.5, "reasoning_type": "other"}
            errors += 1

        llm_results.append(result)

        # Progress + checkpoint every 500
        if (i + 1) % 500 == 0 or i == len(dev_sample) - 1:
            elapsed = time.time() - t0
            rate = (i - start_from + 1) / elapsed
            remaining = (len(dev_sample) - i - 1) / max(rate, 0.1)
            print(f'  [{i+1}/{len(dev_sample)}] {rate:.1f} req/s, '
                  f'~{remaining:.0f}s remaining, {errors} errors')
            # Save checkpoint
            with open(CACHE_FILE, 'w') as f:
                json_mod.dump(llm_results, f)

    print(f'\nDone: {len(llm_results)} results, {errors} errors, '
          f'{time.time()-t0:.1f}s total')

# Quick distribution check
from collections import Counter
impact_dist = Counter(r.get('impact_level', 'low') for r in llm_results)
direction_dist = Counter(r.get('direction', 'neutral') for r in llm_results)
reasoning_dist = Counter(r.get('reasoning_type', 'other') for r in llm_results)
print(f'\nLLM output distributions:')
print(f'  Impact:    {dict(impact_dist)}')
print(f'  Direction: {dict(direction_dist)}')
print(f'  Reasoning: {dict(reasoning_dist)}')
print(f'  Avg confidence: {np.mean([r.get("confidence", 0.5) for r in llm_results]):.3f}')


In [ ]:
# ══════════════════════════════════════════════════════════
# PHASE 2b: LLM vs FinBERT Feature Evaluation
# ══════════════════════════════════════════════════════════
# 5-fold CV on dev-holdout: compare FinBERT 3-dim vs LLM 10-dim vs Combined
# Also: check if LLM impact_level="high" subset has better AUC

from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

print('='*70)
print('PHASE 2b: LLM vs FinBERT FEATURE COMPARISON')
print('='*70)

# --- Encode LLM features (10-dim) ---
impact_map = {'high': [1,0,0], 'medium': [0,1,0], 'low': [0,0,1]}
direction_map = {'positive': 1.0, 'negative': -1.0, 'neutral': 0.0}
reasoning_map = {'earnings': [1,0,0,0,0], 'macro': [0,1,0,0,0],
                 'sentiment': [0,0,1,0,0], 'technical': [0,0,0,1,0],
                 'other': [0,0,0,0,1]}

llm_features = []
for r in llm_results:
    feat = []
    feat.extend(impact_map.get(r.get('impact_level', 'low'), [0,0,1]))  # 3-dim
    feat.append(direction_map.get(r.get('direction', 'neutral'), 0.0))   # 1-dim
    feat.append(float(r.get('confidence', 0.5)))                         # 1-dim
    feat.extend(reasoning_map.get(r.get('reasoning_type', 'other'), [0,0,0,0,1]))  # 5-dim
    llm_features.append(feat)
llm_features = np.array(llm_features, dtype=np.float32)  # (N, 10)
print(f'LLM features: {llm_features.shape}')

# --- Get FinBERT features for same events ---
orig_indices = dev_sample['_orig_idx'].values
emb_orig = np.load(PARAMS['emb_file'], mmap_mode='r')
sent_orig = np.load(PARAMS['sent_file'], mmap_mode='r')

finbert_sent = sent_orig[orig_indices].astype(np.float32)  # (N, 3)
finbert_emb = emb_orig[orig_indices].astype(np.float32)    # (N, 768)
print(f'FinBERT sentiment: {finbert_sent.shape}')
print(f'FinBERT embedding: {finbert_emb.shape}')

# --- Labels ---
y = dev_sample['label'].values
print(f'Labels: {len(y)}, pos_rate={y.mean():.4f}')

# --- 5-fold CV comparison ---
feature_sets = {
    'FinBERT sent (3d)':     finbert_sent,
    'LLM structured (10d)':  llm_features,
    'Combined (13d)':        np.concatenate([finbert_sent, llm_features], axis=1),
    'FinBERT emb (768d)':    finbert_emb,
    'LLM + emb (778d)':      np.concatenate([finbert_emb, llm_features], axis=1),
}

cv_results = {}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

for name, X in feature_sets.items():
    aucs = []
    for fold, (train_i, test_i) in enumerate(skf.split(X, y)):
        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X[train_i])
        X_te = scaler.transform(X[test_i])

        clf = LogisticRegression(max_iter=500, random_state=SEED, C=1.0)
        clf.fit(X_tr, y[train_i])
        prob = clf.predict_proba(X_te)[:, 1]
        aucs.append(roc_auc_score(y[test_i], prob))

    mean_auc = np.mean(aucs)
    std_auc = np.std(aucs)
    cv_results[name] = (mean_auc, std_auc)
    print(f'  {name:<25s}  AUC = {mean_auc:.4f} ± {std_auc:.4f}')

# --- High-impact subset analysis ---
print(f'\n{"="*70}')
print('IMPACT-LEVEL SUBSET ANALYSIS')
print(f'{"="*70}')

impact_levels = [r.get('impact_level', 'low') for r in llm_results]
for level in ['high', 'medium', 'low']:
    mask = np.array([il == level for il in impact_levels])
    n = mask.sum()
    if n < 50:
        print(f'  {level:>8s}: n={n} (too few)')
        continue
    y_sub = y[mask]
    pos_rate = y_sub.mean()
    # Use FinBERT sent + LLM features for this subset
    X_sub = np.concatenate([finbert_sent[mask], llm_features[mask]], axis=1)
    if len(np.unique(y_sub)) < 2:
        print(f'  {level:>8s}: n={n}, pos_rate={pos_rate:.3f} (single class)')
        continue
    # Simple train/test split for subset
    from sklearn.model_selection import cross_val_score
    clf_sub = LogisticRegression(max_iter=500, random_state=SEED, C=1.0)
    scores = cross_val_score(clf_sub, StandardScaler().fit_transform(X_sub),
                              y_sub, cv=min(5, n // 20), scoring='roc_auc')
    print(f'  {level:>8s}: n={n:,} ({n/len(y)*100:.1f}%), '
          f'pos_rate={pos_rate:.3f}, AUC={scores.mean():.4f}±{scores.std():.4f}')

# --- LLM direction accuracy ---
print(f'\n{"="*70}')
print('LLM DIRECTION PREDICTION ACCURACY')
print(f'{"="*70}')
llm_dir = np.array([direction_map.get(r.get('direction', 'neutral'), 0.0)
                     for r in llm_results])
# Positive direction → predict label=1
dir_pred = (llm_dir > 0).astype(int)
dir_mask = llm_dir != 0  # exclude neutral
if dir_mask.sum() > 0:
    dir_acc = (dir_pred[dir_mask] == y[dir_mask]).mean()
    print(f'  Non-neutral predictions: {dir_mask.sum():,} / {len(y):,} '
          f'({dir_mask.sum()/len(y)*100:.1f}%)')
    print(f'  Direction accuracy: {dir_acc:.4f} (random = 0.50)')
    # High-impact + non-neutral
    hi_mask = np.array([il == 'high' for il in impact_levels]) & dir_mask
    if hi_mask.sum() > 50:
        hi_acc = (dir_pred[hi_mask] == y[hi_mask]).mean()
        print(f'  High-impact + directional: n={hi_mask.sum():,}, accuracy={hi_acc:.4f}')

# --- Summary ---
print(f'\n{"="*70}')
print('PHASE 2 SUMMARY')
print(f'{"="*70}')
finbert_auc = cv_results['FinBERT sent (3d)'][0]
llm_auc = cv_results['LLM structured (10d)'][0]
combined_auc = cv_results['Combined (13d)'][0]
delta = llm_auc - finbert_auc
print(f'  FinBERT sentiment AUC:  {finbert_auc:.4f}')
print(f'  LLM structured AUC:    {llm_auc:.4f}')
print(f'  Combined AUC:          {combined_auc:.4f}')
print(f'  LLM vs FinBERT delta:  {delta:+.4f}')
print()
if delta > 0.02:
    print(f'  → LLM provides significant incremental signal (+{delta:.4f})')
    print(f'  → Worth full-scale run (~$19 for 300K stock-days)')
elif delta > 0.005:
    print(f'  → LLM provides marginal signal (+{delta:.4f})')
    print(f'  → Consider cost-benefit before full-scale run')
else:
    print(f'  → LLM provides NO incremental signal over FinBERT')
    print(f'  → Skip full-scale LLM run, save $19')


In [ ]:
# === Results Analysis & Visualization ===
import matplotlib.pyplot as plt

results_df = pd.DataFrame(results)
print('\n' + '='*70)
print('EXPERIMENT RESULTS')
print('='*70)
print(results_df.to_string(index=False, float_format='%.4f'))

# Save results
results_df.to_csv('experiments/phase_c_results.csv', index=False)
print('\nSaved: experiments/phase_c_results.csv')

# --- Bar chart: Val & Test AUC comparison ---
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(results_df))
w = 0.35
ax.bar(x - w/2, results_df['val_auc'], w, label='Val AUC', color='steelblue')
ax.bar(x + w/2, results_df['test_auc'], w, label='Test AUC', color='coral')
ax.set_xticks(x)
ax.set_xticklabels(results_df['experiment'], rotation=30, ha='right', fontsize=9)
ax.set_ylabel('AUC-ROC')
ax.set_title('Phase C: Model Comparison')
ax.legend()
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Random')
ax.set_ylim(0.45, max(results_df[['val_auc','test_auc']].max()) + 0.05)
for i, row in results_df.iterrows():
    ax.text(i - w/2, row['val_auc'] + 0.005, f"{row['val_auc']:.3f}", ha='center', fontsize=8)
    ax.text(i + w/2, row['test_auc'] + 0.005, f"{row['test_auc']:.3f}", ha='center', fontsize=8)
plt.tight_layout()
plt.savefig('plots/phase_c_model_comparison.png', dpi=150, bbox_inches='tight')
print('Saved: plots/phase_c_model_comparison.png')
plt.show()

# --- Ablation delta chart ---
gnn_rows = results_df[results_df['experiment'].str.startswith(('A', 'F'))].copy()
if len(gnn_rows) > 1:
    baseline_auc = gnn_rows.iloc[0]['test_auc']  # A1 as GNN baseline
    gnn_rows['delta'] = gnn_rows['test_auc'] - baseline_auc
    fig, ax = plt.subplots(figsize=(8, 5))
    colors = ['gray'] + ['green' if d > 0 else 'red' for d in gnn_rows['delta'].iloc[1:]]
    ax.bar(gnn_rows['experiment'], gnn_rows['delta'], color=colors)
    ax.set_ylabel('Test AUC delta (vs A1 baseline)')
    ax.set_title('Ablation: Effect of Each Edge Type')
    ax.axhline(y=0, color='black', linewidth=0.5)
    for i, (_, row) in enumerate(gnn_rows.iterrows()):
        ax.text(i, row['delta'] + 0.002, f"+{row['delta']:.4f}" if row['delta'] > 0 else f"{row['delta']:.4f}",
                ha='center', fontsize=9)
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.savefig('plots/phase_c_ablation_delta.png', dpi=150, bbox_inches='tight')
    print('Saved: plots/phase_c_ablation_delta.png')
    plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════
# D.1: DATA-LEVEL DIAGNOSTICS  (no model needed, uses raw data)
# ══════════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

returns_raw = events['return_next'].values  # raw next-day returns (float)

# ── Analysis 1: Label noise quantification ────────────────────────
print('='*60)
print('ANALYSIS 1: Label Noise Quantification')
print('='*60)
print(f'  return_next  mean={np.nanmean(returns_raw):.5f}  std={np.nanstd(returns_raw):.4f}'
      f'  median={np.nanmedian(returns_raw):.5f}')

# What fraction of events have tiny returns (essentially coin flip)?
abs_ret = np.abs(returns_raw)
tiny_pct = (abs_ret < 0.005).mean()   # |ret| < 0.5%
small_pct = (abs_ret < 0.01).mean()   # |ret| < 1%
print(f'  |return| < 0.5%: {tiny_pct:.1%}  (these labels are near-random noise)')
print(f'  |return| < 1.0%: {small_pct:.1%}')

# Bucket analysis
bins = [0, 0.005, 0.01, 0.02, 0.05, np.inf]
bin_labels_name = ['<0.5%', '0.5-1%', '1-2%', '2-5%', '>5%']
bucket = pd.cut(abs_ret, bins=bins, labels=bin_labels_name)
bucket_df = pd.DataFrame({'bucket': bucket, 'label': labels, 'return': returns_raw})
bucket_stats = bucket_df.groupby('bucket', observed=False).agg(
    count=('label', 'size'),
    pos_rate=('label', 'mean'),
    mean_abs_ret=('return', lambda x: np.abs(x).mean()),
).reset_index()
print('\n  Return magnitude buckets:')
print(bucket_stats.to_string(index=False))

# ── Analysis 2: Sentiment confidence vs label alignment ──────────
print('\n' + '='*60)
print('ANALYSIS 2: FinBERT Sentiment Confidence vs Actual Return')
print('='*60)

# sent columns: [pos, neg, neu]
sent_arr = sent[:len(events)].astype(np.float32) if valid_idx is None else sent[valid_idx].astype(np.float32) if valid_idx is not None and len(valid_idx) == len(events) else sent[:len(events)].astype(np.float32)
pos_prob = sent_arr[:, 0]
neg_prob = sent_arr[:, 1]
neu_prob = sent_arr[:, 2]

# Classify sentiment
sent_class = np.where(pos_prob > neg_prob, 'positive', 'negative')
sent_class = np.where(neu_prob > np.maximum(pos_prob, neg_prob), 'neutral', sent_class)
max_conf = np.maximum(pos_prob, neg_prob)  # non-neutral confidence

print(f'  FinBERT sentiment: {(sent_class == "positive").mean():.1%} pos, '
      f'{(sent_class == "negative").mean():.1%} neg, '
      f'{(sent_class == "neutral").mean():.1%} neu')

# Confidence buckets
conf_bins = [0, 0.3, 0.5, 0.7, 1.01]
conf_labels = ['<0.3', '0.3-0.5', '0.5-0.7', '>0.7']
conf_bucket = pd.cut(max_conf, bins=conf_bins, labels=conf_labels)
conf_df = pd.DataFrame({
    'conf_bucket': conf_bucket, 'sent_class': sent_class,
    'label': labels, 'return': returns_raw,
})

# For positive-sentiment news: what's the actual positive label rate?
# For negative-sentiment news: what's the actual negative label rate?
for sc in ['positive', 'negative']:
    sub = conf_df[conf_df['sent_class'] == sc]
    tbl = sub.groupby('conf_bucket', observed=False).agg(
        count=('label', 'size'),
        actual_alignment=('label', lambda x: x.mean() if sc == 'positive' else 1 - x.mean()),
        mean_return=('return', 'mean'),
    ).reset_index()
    print(f'\n  {sc.upper()} sentiment news:')
    print(f'  (alignment = fraction where sentiment matches actual direction)')
    print(tbl.to_string(index=False))

# ── Analysis 3: Per-sector label distribution ────────────────────
print('\n' + '='*60)
print('ANALYSIS 3: Per-Sector Statistics')
print('='*60)

events_sec = events.copy()
events_sec['sector'] = events_sec['ticker'].map(sector_map).fillna('Unknown')

sector_stats = events_sec.groupby('sector').agg(
    n_events=('label', 'size'),
    pos_rate=('label', 'mean'),
    mean_return=('return_next', 'mean'),
    std_return=('return_next', 'std'),
    n_tickers=('ticker', 'nunique'),
).sort_values('n_events', ascending=False)
print(sector_stats.to_string())

# ── Analysis 4: Temporal stability ───────────────────────────────
print('\n' + '='*60)
print('ANALYSIS 4: Temporal Stability (Quarterly)')
print('='*60)

events_sec['quarter'] = events_sec['date'].dt.to_period('Q').astype(str)
qtr_stats = events_sec.groupby('quarter').agg(
    n_events=('label', 'size'),
    pos_rate=('label', 'mean'),
    mean_return=('return_next', 'mean'),
).reset_index()
print(qtr_stats.to_string(index=False))

# ── Plots ─────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 14))
gs = GridSpec(2, 2, hspace=0.35, wspace=0.3)

# Plot 1: Return distribution
ax1 = fig.add_subplot(gs[0, 0])
ax1.hist(returns_raw[~np.isnan(returns_raw)], bins=200, range=(-0.1, 0.1),
         color='steelblue', alpha=0.8, edgecolor='none')
ax1.axvline(x=0, color='red', linestyle='--', linewidth=1)
ax1.axvline(x=0.005, color='orange', linestyle=':', linewidth=1, label='±0.5%')
ax1.axvline(x=-0.005, color='orange', linestyle=':', linewidth=1)
ax1.set_xlabel('Next-Day Return')
ax1.set_ylabel('Count')
ax1.set_title(f'Return Distribution\n({tiny_pct:.0%} of events have |return| < 0.5%)')
ax1.legend()

# Plot 2: Sentiment alignment by confidence
ax2 = fig.add_subplot(gs[0, 1])
align_data = []
for sc in ['positive', 'negative']:
    sub = conf_df[conf_df['sent_class'] == sc]
    for cl in conf_labels:
        mask = sub['conf_bucket'] == cl
        if mask.sum() > 100:
            rate = sub.loc[mask, 'label'].mean() if sc == 'positive' else (1 - sub.loc[mask, 'label'].mean())
            align_data.append({'confidence': cl, 'sentiment': sc, 'alignment': rate, 'n': mask.sum()})
align_df = pd.DataFrame(align_data)
for i, sc in enumerate(['positive', 'negative']):
    sub = align_df[align_df['sentiment'] == sc]
    offset = -0.2 + i * 0.4
    bars = ax2.bar(np.arange(len(sub)) + offset, sub['alignment'], width=0.35,
                   label=f'{sc} news', color=['forestgreen', 'firebrick'][i], alpha=0.8)
    for j, (_, row) in enumerate(sub.iterrows()):
        ax2.text(j + offset, row['alignment'] + 0.005, f"n={row['n']//1000}k",
                 ha='center', fontsize=7)
ax2.set_xticks(range(len(conf_labels)))
ax2.set_xticklabels(conf_labels)
ax2.set_xlabel('FinBERT Confidence (max of pos/neg prob)')
ax2.set_ylabel('Alignment Rate')
ax2.set_title('Sentiment-Direction Alignment\n(0.5 = random, >0.5 = signal)')
ax2.axhline(y=0.5, color='gray', linestyle='--', alpha=0.7)
ax2.legend()

# Plot 3: Per-sector stats
ax3 = fig.add_subplot(gs[1, 0])
ss = sector_stats.sort_values('pos_rate')
colors = ['firebrick' if x < 0.49 else 'forestgreen' if x > 0.51 else 'gray'
          for x in ss['pos_rate']]
ax3.barh(ss.index, ss['pos_rate'], color=colors, alpha=0.8)
ax3.axvline(x=0.5, color='gray', linestyle='--', alpha=0.7)
ax3.set_xlabel('Positive Label Rate')
ax3.set_title('Label Distribution by Sector')
for i, (sec, row) in enumerate(ss.iterrows()):
    ax3.text(row['pos_rate'] + 0.001, i, f"{row['n_events']//1000}k",
             va='center', fontsize=8)

# Plot 4: Temporal stability
ax4 = fig.add_subplot(gs[1, 1])
ax4.plot(qtr_stats['quarter'], qtr_stats['pos_rate'], 'o-', color='steelblue',
         markersize=4, label='Positive label rate')
ax4.axhline(y=0.5, color='gray', linestyle='--', alpha=0.7)
ax4.set_xlabel('Quarter')
ax4.set_ylabel('Positive Rate')
ax4.set_title('Label Balance Over Time')
ax4.tick_params(axis='x', rotation=45)
ax4.legend()

# Add event count on twin axis
ax4b = ax4.twinx()
ax4b.bar(qtr_stats['quarter'], qtr_stats['n_events'] / 1000, alpha=0.15,
         color='steelblue', label='Events (k)')
ax4b.set_ylabel('Events (thousands)')

plt.suptitle('D.1: Data-Level Diagnostics', fontsize=14, fontweight='bold', y=1.01)
plt.savefig('plots/phase_c_diagnostics_data.png', dpi=150, bbox_inches='tight')
print('\nSaved: plots/phase_c_diagnostics_data.png')
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════
# D.2: MODEL PREDICTION DIAGNOSTICS (retrain LR, slice AUC by dimensions)
# ══════════════════════════════════════════════════════════════════
# Re-train B1 (LR + FinBERT) to get prediction scores for fine-grained analysis.
# This cell is self-contained — it only uses variables from earlier cells.

from sklearn.linear_model import SGDClassifier
from sklearn.metrics import roc_auc_score, accuracy_score

print('Re-training LR + FinBERT for diagnostic scores...')
diag_clf = SGDClassifier(loss='log_loss', random_state=SEED, learning_rate='optimal',
                         alpha=1e-4, max_iter=1)
CHUNK = 50000
for epoch in range(5):
    shuffled = train_idx.copy()
    np.random.shuffle(shuffled)
    for i in range(0, len(shuffled), CHUNK):
        idx = shuffled[i:i+CHUNK]
        X = emb[idx].astype(np.float32)
        y = labels[idx]
        diag_clf.partial_fit(X, y, classes=[0, 1])

# Get scores for val + test
test_scores = diag_clf.decision_function(emb[test_idx].astype(np.float32))
val_scores = diag_clf.decision_function(emb[val_idx].astype(np.float32))
overall_test_auc = roc_auc_score(labels[test_idx], test_scores)
print(f'LR overall test AUC: {overall_test_auc:.4f}')

# Prepare test set metadata for slicing
test_events = events.iloc[test_idx].copy().reset_index(drop=True)
test_events['score'] = test_scores
test_events['true_label'] = labels[test_idx]
test_events['sector'] = test_events['ticker'].map(sector_map).fillna('Unknown')

# FinBERT sentiment for test set
test_sent = sent[test_idx].astype(np.float32) if valid_idx is None else sent[test_idx].astype(np.float32)
test_events['pos_prob'] = test_sent[:, 0]
test_events['neg_prob'] = test_sent[:, 1]
test_events['neu_prob'] = test_sent[:, 2]
test_events['max_conf'] = np.maximum(test_sent[:, 0], test_sent[:, 1])
test_events['abs_return'] = np.abs(test_events['return_next'])

# ── Analysis 5: Prediction score distribution ─────────────────────
print('\n' + '='*60)
print('ANALYSIS 5: Prediction Score Distribution')
print('='*60)
for lab_val, lab_name in [(0, 'Negative'), (1, 'Positive')]:
    sub = test_events[test_events['true_label'] == lab_val]['score']
    print(f'  {lab_name}: mean={sub.mean():.4f} std={sub.std():.4f} '
          f'median={sub.median():.4f}')
separation = (test_events[test_events['true_label']==1]['score'].mean() -
              test_events[test_events['true_label']==0]['score'].mean())
print(f'  Mean separation: {separation:.5f}  (larger = more discriminative)')

# ── Analysis 6: Per-sector AUC ────────────────────────────────────
print('\n' + '='*60)
print('ANALYSIS 6: Per-Sector AUC (Test Set, LR + FinBERT)')
print('='*60)
sector_auc = []
for sec in sorted(test_events['sector'].unique()):
    sub = test_events[test_events['sector'] == sec]
    if len(sub) > 100 and sub['true_label'].nunique() == 2:
        auc = roc_auc_score(sub['true_label'], sub['score'])
        sector_auc.append({'sector': sec, 'n_events': len(sub), 'auc': auc,
                          'pos_rate': sub['true_label'].mean()})
sector_auc_df = pd.DataFrame(sector_auc).sort_values('auc', ascending=False)
print(sector_auc_df.to_string(index=False, float_format='%.4f'))
print(f'\n  Best: {sector_auc_df.iloc[0]["sector"]} ({sector_auc_df.iloc[0]["auc"]:.4f})')
print(f'  Worst: {sector_auc_df.iloc[-1]["sector"]} ({sector_auc_df.iloc[-1]["auc"]:.4f})')

# ── Analysis 7: AUC by sentiment confidence ──────────────────────
print('\n' + '='*60)
print('ANALYSIS 7: AUC by Sentiment Confidence')
print('='*60)
conf_auc = []
for lo, hi, name in [(0, 0.3, '<0.3'), (0.3, 0.5, '0.3-0.5'),
                      (0.5, 0.7, '0.5-0.7'), (0.7, 1.01, '>0.7')]:
    sub = test_events[(test_events['max_conf'] >= lo) & (test_events['max_conf'] < hi)]
    if len(sub) > 100 and sub['true_label'].nunique() == 2:
        auc = roc_auc_score(sub['true_label'], sub['score'])
        conf_auc.append({'confidence': name, 'n_events': len(sub), 'auc': auc})
conf_auc_df = pd.DataFrame(conf_auc)
print(conf_auc_df.to_string(index=False, float_format='%.4f'))

# Also: AUC for non-neutral only vs all
non_neutral = test_events[test_events['max_conf'] > 0.5]
if len(non_neutral) > 100 and non_neutral['true_label'].nunique() == 2:
    auc_nn = roc_auc_score(non_neutral['true_label'], non_neutral['score'])
    print(f'\n  Non-neutral events (conf>0.5): {len(non_neutral):,} events, AUC={auc_nn:.4f}')
    print(f'  vs All events: {len(test_events):,} events, AUC={overall_test_auc:.4f}')

# ── Analysis 8: AUC by return magnitude ──────────────────────────
print('\n' + '='*60)
print('ANALYSIS 8: AUC by Return Magnitude')
print('='*60)
ret_auc = []
for lo, hi, name in [(0, 0.005, '<0.5%'), (0.005, 0.01, '0.5-1%'),
                      (0.01, 0.02, '1-2%'), (0.02, 0.05, '2-5%'),
                      (0.05, np.inf, '>5%')]:
    sub = test_events[(test_events['abs_return'] >= lo) & (test_events['abs_return'] < hi)]
    if len(sub) > 100 and sub['true_label'].nunique() == 2:
        auc = roc_auc_score(sub['true_label'], sub['score'])
        ret_auc.append({'return_bucket': name, 'n_events': len(sub), 'auc': auc,
                       'pos_rate': sub['true_label'].mean()})
ret_auc_df = pd.DataFrame(ret_auc)
print(ret_auc_df.to_string(index=False, float_format='%.4f'))

# Key test: large-move events only
large_move = test_events[test_events['abs_return'] > 0.01]
if len(large_move) > 100 and large_move['true_label'].nunique() == 2:
    auc_lm = roc_auc_score(large_move['true_label'], large_move['score'])
    print(f'\n  Large-move events (|ret|>1%): {len(large_move):,} events, AUC={auc_lm:.4f}')
    print(f'  vs All events: {len(test_events):,} events, AUC={overall_test_auc:.4f}')

# ── Plots ─────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 14))
gs = GridSpec(2, 2, hspace=0.35, wspace=0.3)

# Plot 5: Score distribution
ax5 = fig.add_subplot(gs[0, 0])
for lab_val, lab_name, color in [(0, 'Negative (down)', 'firebrick'),
                                  (1, 'Positive (up)', 'forestgreen')]:
    sub = test_events[test_events['true_label'] == lab_val]['score']
    ax5.hist(sub, bins=100, alpha=0.5, color=color, label=lab_name, density=True)
ax5.set_xlabel('LR Decision Function Score')
ax5.set_ylabel('Density')
ax5.set_title(f'Prediction Score by True Label\n(separation={separation:.5f})')
ax5.legend()

# Plot 6: Per-sector AUC
ax6 = fig.add_subplot(gs[0, 1])
sa = sector_auc_df.sort_values('auc')
colors = ['forestgreen' if x > 0.52 else 'firebrick' if x < 0.48 else 'gray'
          for x in sa['auc']]
ax6.barh(sa['sector'], sa['auc'], color=colors, alpha=0.8)
ax6.axvline(x=0.5, color='gray', linestyle='--', alpha=0.7, label='Random')
ax6.set_xlabel('Test AUC')
ax6.set_title('Per-Sector AUC (LR + FinBERT)')
for i, (_, row) in enumerate(sa.iterrows()):
    ax6.text(row['auc'] + 0.002, i, f"{row['auc']:.3f} ({row['n_events']//1000}k)",
             va='center', fontsize=8)
ax6.set_xlim(min(sa['auc'].min() - 0.03, 0.45), max(sa['auc'].max() + 0.05, 0.55))

# Plot 7: AUC by sentiment confidence
ax7 = fig.add_subplot(gs[1, 0])
if len(conf_auc_df) > 0:
    colors = ['forestgreen' if x > 0.52 else 'firebrick' if x < 0.48 else 'gray'
              for x in conf_auc_df['auc']]
    bars = ax7.bar(conf_auc_df['confidence'], conf_auc_df['auc'], color=colors, alpha=0.8)
    ax7.axhline(y=0.5, color='gray', linestyle='--', alpha=0.7)
    ax7.set_xlabel('FinBERT Confidence Bucket')
    ax7.set_ylabel('Test AUC')
    ax7.set_title('AUC by Sentiment Confidence')
    for i, (_, row) in enumerate(conf_auc_df.iterrows()):
        ax7.text(i, row['auc'] + 0.003, f"{row['auc']:.3f}\n({row['n_events']//1000}k)",
                 ha='center', fontsize=8)
    ax7.set_ylim(0.45, max(conf_auc_df['auc'].max() + 0.05, 0.55))

# Plot 8: AUC by return magnitude
ax8 = fig.add_subplot(gs[1, 1])
if len(ret_auc_df) > 0:
    colors = ['forestgreen' if x > 0.52 else 'firebrick' if x < 0.48 else 'gray'
              for x in ret_auc_df['auc']]
    bars = ax8.bar(ret_auc_df['return_bucket'], ret_auc_df['auc'], color=colors, alpha=0.8)
    ax8.axhline(y=0.5, color='gray', linestyle='--', alpha=0.7)
    ax8.set_xlabel('|Next-Day Return| Bucket')
    ax8.set_ylabel('Test AUC')
    ax8.set_title('AUC by Return Magnitude\n(Can news predict large moves?)')
    for i, (_, row) in enumerate(ret_auc_df.iterrows()):
        ax8.text(i, row['auc'] + 0.003, f"{row['auc']:.3f}\n({row['n_events']//1000}k)",
                 ha='center', fontsize=8)
    ax8.set_ylim(0.45, max(ret_auc_df['auc'].max() + 0.05, 0.55))

plt.suptitle('D.2: Model Prediction Diagnostics (LR + FinBERT on Test Set)',
             fontsize=14, fontweight='bold', y=1.01)
plt.savefig('plots/phase_c_diagnostics_model.png', dpi=150, bbox_inches='tight')
print('\nSaved: plots/phase_c_diagnostics_model.png')
plt.show()

# ── Summary Table ─────────────────────────────────────────────────
print('\n' + '='*60)
print('DIAGNOSTIC SUMMARY')
print('='*60)
print(f'Overall LR Test AUC: {overall_test_auc:.4f}')
if len(sector_auc_df) > 0:
    print(f'Sector AUC range: {sector_auc_df["auc"].min():.4f} - {sector_auc_df["auc"].max():.4f}')
if len(conf_auc_df) > 0:
    print(f'Confidence AUC range: {conf_auc_df["auc"].min():.4f} - {conf_auc_df["auc"].max():.4f}')
if len(ret_auc_df) > 0:
    print(f'Return-mag AUC range: {ret_auc_df["auc"].min():.4f} - {ret_auc_df["auc"].max():.4f}')
print('\nKey questions answered:')
print('  1. Is any sector predictable? → Check sector AUC chart')
print('  2. Does high-confidence sentiment help? → Check confidence AUC chart')
print('  3. Can news predict large moves? → Check return magnitude AUC chart')
print('  4. How much label noise? → Check return distribution histogram')

## Observations & Next Steps

### Phase C v1 Results (AUC ≈ 0.50, near-random)

| Experiment | Val AUC | Test AUC |
|-----------|---------|----------|
| B1: LR + FinBERT | 0.5018 | 0.4976 |
| B2: LR + Sentiment | 0.5044 | 0.5027 |
| A1: GNN news→stock only | 0.5085 | 0.4913 |
| A2: + correlation | 0.5122 | 0.4949 |
| A3: + sector | 0.5133 | 0.4961 |
| Full: all edges | 0.5133 | 0.5069 |

**Core finding:** Neither FinBERT embeddings nor sentiment scores carry predictive signal for raw next-day returns. GNN graph structure provides marginal lift (+1.6% test AUC vs A1) but cannot overcome zero-signal features.

### Diagnostic Analysis (D.1 & D.2)

*(Fill in after running D.1 and D.2 on Colab)*

**D.1 Data-Level:**
- Label noise (% events with |return| < 0.5%): ?
- Sentiment-direction alignment: ?
- Sector-level imbalance: ?
- Temporal drift: ?

**D.2 Model Prediction:**
- Score distribution separation: ?
- Best/worst sector AUC: ?
- High-confidence sentiment AUC: ?
- Large-move (|ret|>1%) AUC: ?

### Implications for Next Steps

Based on diagnostics, potential improvements:
1. **Market-adjusted labels** — remove market beta from returns
2. **High-confidence event filtering** — only keep FinBERT non-neutral events
3. **Richer stock features** — add rolling price statistics
4. **Multi-day return window** — use 3-day cumulative return